# SmartCrop — Notebook 05: Model Training

**Target:** `yield_t_ha`  
**Candidates:** linear regression, decision tree, random forest, gradient boosting.  
XGBoost is not used. It is not required for this comparison.


## 1. Introduction

Training estimates maize yield per hectare from weather, soil, and management inputs. The saved object is a scikit-learn pipeline: feature builder, median imputer, and the regressor. The API loads that object. It does not fit a new model when a user asks for a prediction.


## 2. Objectives

1. Split seasons in time.
2. Fit four regressors on the earlier seasons.
3. Record training time, features, hyperparameters, validation metrics, and grouped cross-validation.
4. Save `reports/model_reports/model_comparison.csv`.


## 3. Dataset Description

The modelling frame is the clean synthetic file. Raw inputs are rainfall, three temperatures, soil pH, soil nitrogen, soil organic carbon, fertilizer, and planting day. The pipeline adds the three engineered columns from Notebook 03.


## 4. Methodology

Years are ordered. The earliest seasons train. The third-from-last season validates. The last two seasons are held out and are not used in this notebook.

The imputer learns medians on the training rows inside the pipeline. Grouped cross-validation splits the training window by season, so a whole season is held out of each fold. The test seasons never enter those folds.

Model choice, in the next notebooks, uses validation R². Lower validation RMSE breaks a tie.


## 5. Implementation


In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    if (start / "data-science").is_dir():
        candidates.insert(0, start / "data-science")
    for candidate in candidates:
        if (candidate / "src" / "data" / "schema.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the SmartCrop repository.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 120)

from data.schema import SYNTHETIC_DISCLAIMER
from models.training import RANDOM_SEED, run_model_comparison

print(SYNTHETIC_DISCLAIMER)
clean = pd.read_csv(ROOT / "data" / "processed" / "clean_dataset.csv")
comparison, details = run_model_comparison(clean, RANDOM_SEED)
out = ROOT / "reports" / "model_reports"
out.mkdir(parents=True, exist_ok=True)
comparison.to_csv(out / "model_comparison.csv", index=False)
comparison.to_excel(out / "model_comparison.xlsx", index=False)
print("Train years:", details["masks"]["train_years"])
print("Validation years:", details["masks"]["validation_years"])
print("Held-out test years (not scored here):", details["masks"]["test_years"])
display(comparison.drop(columns=["hyperparameters", "features"]))


SYNTHETIC DATA. These rows were generated by the SmartCrop synthetic generator for software and model-development exercises. They are not measurements from Macheke, Mashonaland East, Zimbabwe, and they must not be cited as agricultural statistics.


Train years: [2017, 2018, 2019, 2020, 2021]
Validation years: [2022]
Held-out test years (not scored here): [2023, 2024]


,model_name,training_seconds,validation_mae,validation_rmse,validation_r2,cv_mae,cv_rmse,cv_r2,train_rows,validation_rows
0,Random Forest Regressor,0.2596,0.2850,0.3516,-0.3097,0.1576,0.1979,0.4812,585,118
1,Gradient Boosting Regressor,0.1365,0.2932,0.3582,-0.3588,0.1542,0.1945,0.4983,585,118
2,Decision Tree Regressor,0.0035,0.2984,0.3706,-0.4551,0.1910,0.2395,0.2377,585,118
3,Linear Regression,0.0135,0.3861,0.4505,-1.1501,0.1784,0.2284,0.3109,585,118


## 6. Results

The comparison table lists each algorithm, its validation MAE, RMSE, and R², and the mean grouped cross-validation scores on the training window. R² is the coefficient of determination. It is not accuracy.


## 7. Visualizations

Training comparison is a table. Residual and importance plots belong to the evaluation notebook, after a model has been selected.


## 8. Interpretation

A higher validation R² means the model explained more of the yield variation in the validation season. That season is still synthetic. A fast linear model can be preferred only when its validation score leads. Hyperparameters are fixed and documented in the saved table. They were not searched on the test seasons.


## 9. Conclusion

Four models have validation scores. Notebook 06 evaluates the selection rule and the held-out seasons. Notebook 07 writes the single serialized model.
